In [10]:
# !pip install sentence-transformers
# !pip install hf_xet

In [11]:
import torch
from sentence_transformers import SentenceTransformer, util

In [12]:
# (A) 다목적 한국어 SBERT
# model_name = "jhgan/ko-sroberta-multitask"

# (B) 문장 유사도 특화 (SimCSE-한국어 멀티태스크)
model_name = "BM-K/KoSimCSE-roberta-multitask"

# (C) NLI/STS로 학습된 한국어 SBERT
# model_name = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"

device = "cuda" if torch.cuda.is_available() else "cpu"
sbert = SentenceTransformer(model_name, device=device)

# (선택) 최대 토큰 길이 조정
sbert.max_seq_length = 256  # 길면 올리고, VRAM 부족하면 줄이세요.


No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [13]:
doc1 = "이 카메라는 색감이 자연스럽고 배터리도 오래가요."
doc2 = "배터리 성능이 좋고 사진 품질이 뛰어납니다."

with torch.inference_mode():
    emb1 = sbert.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sbert.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)

cos_sim = util.cos_sim(emb1, emb2).item()  # -1 ~ 1
print(f"유사도: {cos_sim:.4f}")


유사도: 0.7710


In [14]:
sentences = [
    "삼성전자 주가가 올랐다.",
    "코스피가 상승 마감했다.",
    "비가 많이 내려 항공편이 취소됐다.",
]

with torch.inference_mode():
    embs = sbert.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)

# 모든 쌍 유사도 행렬 (n x n)
sim_matrix = util.cos_sim(embs, embs)
print(sim_matrix)  # 필요시 numpy로 변환해 시각화/랭킹

# 쿼리 한 개와 코퍼스 유사도 TOP-K
query = "증시가 강세였다."
query_emb = sbert.encode(query, convert_to_tensor=True, normalize_embeddings=True)
top_k = 2
hits = torch.topk(util.cos_sim(query_emb, embs).squeeze(0), k=top_k)
for score, idx in zip(hits.values.tolist(), hits.indices.tolist()):
    print(f"{score:.4f} | {sentences[idx]}")


tensor([[1.0000, 0.4075, 0.0027],
        [0.4075, 1.0000, 0.1047],
        [0.0027, 0.1047, 1.0000]])
0.6451 | 삼성전자 주가가 올랐다.
0.6370 | 코스피가 상승 마감했다.


In [15]:
# pip install sentence-transformers scikit-learn pandas torch
import re
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"


In [16]:
def normalize_korean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = re.sub(r"[^0-9a-zA-Z가-힣ㄱ-ㅎㅏ-ㅣ .,!?\"'’‘…~\-]", " ", text)  # 특수문자 제거
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [17]:

# 1) 데이터 로드 ----------------------------------------------------
PATH = "data/ratings_train.txt"
df = pd.read_csv(PATH, sep="\t").dropna(subset=["document", "label"])
# 필요시 빠른 데모용 샘플링 (주석 해제 시)
df = df.sample(n=8000, random_state=42).reset_index(drop=True)
df["document"] = df["document"].apply(normalize_korean_text)

# Train/Test split --------------------------------------------------
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)


In [18]:

# 2) SBERT 모델 로드 ------------------------------------------------
# 한국어용 SBERT 예시 (둘 중 하나 골라 쓰기)
# SentenceTransformer 모델은 미세 튜닝이 어려워서 Trainer를 사용하지 않음
# (A) 다목적 한국어 SBERT
# model_name = "jhgan/ko-sroberta-multitask"

# (B) 문장 유사도 특화 (SimCSE-한국어 멀티태스크)
model_name = "BM-K/KoSimCSE-roberta-multitask"

# (C) NLI/STS로 학습된 한국어 SBERT
# model_name = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"

sbert = SentenceTransformer(model_name, device=device)


No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [19]:

# 3) Dataset 정의 ---------------------------------------------------
class SBERTDataset(Dataset):
    def __init__(self, texts, labels, model, normalize=True):
        with torch.inference_mode():
            self.embs = model.encode(
                texts, convert_to_tensor=True, normalize_embeddings=normalize
            )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return self.embs[i], self.labels[i]

train_ds = SBERTDataset(train_df["document"].tolist(), train_df["label"].tolist(), sbert)
test_ds  = SBERTDataset(test_df["document"].tolist(),  test_df["label"].tolist(), sbert)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=256)


In [20]:

# 4) MLP 분류기 -----------------------------------------------------
class MLPHead(nn.Module):
    def __init__(self, in_dim, hidden=256, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x): return self.net(x)

in_dim = sbert.get_sentence_embedding_dimension()
clf = MLPHead(in_dim).to(device)

crit = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(clf.parameters(), lr=2e-4)


In [21]:

# 5) 학습 -----------------------------------------------------------
clf.train()
for epoch in range(5):
    total = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = clf(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    print(f"epoch {epoch+1}, loss={total/len(train_ds):.4f}")


epoch 1, loss=0.6485
epoch 2, loss=0.5197
epoch 3, loss=0.4442
epoch 4, loss=0.4173
epoch 5, loss=0.4033


In [22]:

# 6) 테스트 ---------------------------------------------------------
clf.eval()
y_true, y_pred = [], []
with torch.inference_mode():
    for xb, yb in test_dl:
        xb = xb.to(device)
        logits = clf(xb)
        pred = logits.argmax(dim=1).cpu().tolist()
        y_pred.extend(pred)
        y_true.extend(yb.tolist())

print("Test Accuracy:", accuracy_score(y_true, y_pred))
print("Test F1:", f1_score(y_true, y_pred, average="macro"))


Test Accuracy: 0.825625
Test F1: 0.8255104230551389


In [24]:
import torch
import numpy as np
from torch.utils.data import DataLoader

# 라벨 맵(원하는대로 수정)
id2label = {0: "부정", 1: "긍정"}

@torch.no_grad()
def predict_reviews(
    texts, 
    batch_size: int = 64, 
    normalize_emb: bool = True,
    apply_normalize_fn: bool = True
):
    """
    texts: str 또는 List[str]
    return: List[dict] - 각 문장별 예측 라벨/확률
    """
    # 단일 문자열도 리스트로 변환
    if isinstance(texts, str):
        texts = [texts]

    # 학습시 사용한 전처리와 동일하게
    if apply_normalize_fn:
        texts_proc = [normalize_korean_text(t) for t in texts]
    else:
        texts_proc = texts

    sbert.eval()
    clf.eval()

    results = []
    # SBERT로 한꺼번에 임베딩(메모리가 작으면 DataLoader로 쪼개서)
    # 여기서는 배치 처리
    for i in range(0, len(texts_proc), batch_size):
        batch_texts = texts_proc[i : i + batch_size]
        embs = sbert.encode(
            batch_texts,
            convert_to_tensor=True,
            normalize_embeddings=normalize_emb,
        ).to(device)  # [B, D]

        logits = clf(embs)                       # [B, num_classes]
        probs = logits.softmax(dim=-1).cpu()     # [B, num_classes]
        preds = probs.argmax(dim=-1).tolist()

        for j, p in enumerate(preds):
            prob = float(probs[j, p])
            results.append({
                "text": texts[i + j],
                "pred_label": id2label[p],
                "prob": prob,
                "probs": probs[j].tolist(),  # 원하면 전체 확률 벡터
            })
    return results

# 사용 예시 ----------------------------------------------------------
samples = [
    "와... 영화 진짜 최고였어요. 또 보고 싶네요!",
    "스토리 엉망이고 연기도 별로였음. 추천 못함.",
    "그럭저럭 볼만했지만 크게 인상적이진 않았어요."
]

out = predict_reviews(samples)
for r in out:
    print(f"[{r['pred_label']}] p={r['prob']:.3f} | {r['text']}")


[긍정] p=0.974 | 와... 영화 진짜 최고였어요. 또 보고 싶네요!
[부정] p=0.977 | 스토리 엉망이고 연기도 별로였음. 추천 못함.
[부정] p=0.719 | 그럭저럭 볼만했지만 크게 인상적이진 않았어요.


In [1]:
# ✅ 설치 (필요 시)
# pip install transformers datasets scikit-learn accelerate torch

import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer, AutoModel,
    Trainer, TrainingArguments, DataCollatorWithPadding
)


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:

# ------------------------------------------------------------
# ✅ 1) 데이터 로드/정리 함수
#    - ratings_train.txt / ratings_test.txt 구조 동일
#    - TSV 구조: id, document, label
#    - label이 없으면 추론용으로 -1을 저장
#    - 공백/결측 문장 제거
# ------------------------------------------------------------

DATA_PATH = "data/ratings_test.txt"
df = pd.read_csv(DATA_PATH, sep="\t")



# ---- 텍스트 전처리 ----
def normalize_korean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = re.sub(r"[^0-9a-zA-Z가-힣ㄱ-ㅎㅏ-ㅣ .,!?\"'’‘…~\-]", " ", text)  # 특수문자 제거
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["document"] = df["document"].map(normalize_korean_text)


# 공백이거나 결측이면 제거
df = df[(df["document"] != "") & (df["document"].notna())]

# label은 0/1이거나 -1(추론용)만 허용
df = df[df["label"].isin([0, 1])]

df =  df[["document", "label"]].reset_index(drop=True)

# 필요시 빠른 데모용 샘플링 (주석 해제 시)
df = df.sample(n=8000, random_state=42).reset_index(drop=True)

# ------------------------------------------------------------
# ✅ 2) Train/Valid 분할 (Stratified Split)
# ------------------------------------------------------------
train_df, valid_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)


In [7]:

# ------------------------------------------------------------
# ✅ 3) 토크나이저/모델 선택
#     - SBERT 구조를 직접 구현할 것이므로
#       HF에서 가져올 수 있는 문장 임베딩용 모델 사용
# ------------------------------------------------------------
# MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODEL_NAME = "BM-K/KoSimCSE-roberta-multitask"
# 또는 한국어 BERT 기반 분류하고 싶으면:
# MODEL_NAME = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ------------------------------------------------------------
# ✅ Dataset 정의
#     - HF Trainer는 모델에 input_ids, attention_mask 등을 넘겨야 하므로
#       토크나이징을 여기서 수행
# ------------------------------------------------------------
class KoSentiDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): 
        return len(self.texts)

    def __getitem__(self, i):
        # ---- 문장 1개 토크나이징 ----
        enc = self.tokenizer(
            self.texts[i],
            truncation=True,
            padding=False,       # collator에서 padding 수행
            max_length=self.max_len,
            return_tensors="pt"
        )
        # enc: {'input_ids': [[...]], 'attention_mask': [[...]]}
        item = {k: v.squeeze(0) for k, v in enc.items()}
        # 라벨 추가
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

# Dataset 생성
train_ds = KoSentiDataset(train_df["document"].tolist(), train_df["label"].tolist(), tokenizer)
valid_ds = KoSentiDataset(valid_df["document"].tolist(), valid_df["label"].tolist(), tokenizer)

# ✅ padding은 collator가 처리
collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [8]:

# ------------------------------------------------------------
# ✅ 4) SBERT의 핵심: Mean Pooling + MLP 분류기
# ------------------------------------------------------------
class SBertPooler(nn.Module):
    """ 
    Mean Pooling:
    - last_hidden_state: [B, T, H]
    - attention_mask:    [B, T]
    - 마스크된 토큰만 평균 내어 문장 임베딩 생성
    """
    def forward(self, token_embeddings: torch.Tensor, attention_mask: torch.Tensor):
        mask = attention_mask.unsqueeze(-1)      # [B, T, 1]
        masked_emb = token_embeddings * mask     # PAD 토큰은 0이 됨
        sum_emb = masked_emb.sum(dim=1)          # 모든 토큰 합
        lengths = mask.sum(dim=1).clamp(min=1e-9)  # 유효 토큰 수 (실제 토큰의 개수(0인 경우 최소값 지정))
        return sum_emb / lengths                 # 평균 벡터


class SBertForSequenceClassification(nn.Module):
    """
    HF AutoModel → Mean Pooling → MLP 분류기
    즉, SBERT 구조를 그대로 HF Trainer에 맞게 커스텀 구현한 모델
    """
    def __init__(self, model_name: str, num_labels: int = 2, hidden: int = 256, p: float = 0.2):
        super().__init__()

        # ✅ HuggingFace 백본 로드 (BERT, RoBERTa 등)
        self.backbone = AutoModel.from_pretrained(model_name)
        hdim = self.backbone.config.hidden_size

        # ✅ SBERT의 핵심 pooling
        self.pooler = SBertPooler()

        # ✅ 비선형 MLP 분류기
        self.classifier = nn.Sequential(
            nn.Linear(hdim, hidden),
            nn.ReLU(),          # 비선형 활성화 → 표현력 증가
            nn.Dropout(p),
            nn.Linear(hidden, num_labels),
        )

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        # ---- Transformer Forward ----
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            # token_type_ids는 모델이 지원할 때만 전달
            token_type_ids=token_type_ids if "token_type_ids" in self.backbone.forward.__code__.co_varnames else None,
            return_dict=True
        )

        # ✅ Mean Pooling → 문장 임베딩 생성
        sent_emb = self.pooler(out.last_hidden_state, attention_mask)  # [B, H]

        # ✅ 분류기 통과
        logits = self.classifier(sent_emb)                             # [B, num_labels]

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

# 모델 생성
model = SBertForSequenceClassification(MODEL_NAME, num_labels=2)


In [9]:

# ------------------------------------------------------------
# ✅ 5) 평가 지표 함수 (Trainer용)
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro")  # 불균형 대응
    return {"accuracy": acc, "f1_macro": f1}

# ------------------------------------------------------------
# ✅ 6) Trainer 설정 & 학습
# ------------------------------------------------------------
args = TrainingArguments(
    output_dir="./sbert-cls-out",

    # 백본까지 미세조정하므로 learning_rate는 작게 유지
    learning_rate=2e-5,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,

    num_train_epochs=3,

    eval_strategy="epoch",   # epoch마다 평가
    save_strategy="epoch",         # epoch마다 체크포인트 저장

    load_best_model_at_end=True,   # 최고 F1 기준으로 모델 로드
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=50,

    fp16=torch.cuda.is_available(),  # GPU에서 FP16 사용
    report_to="none",                # WandB 등 사용 안함
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# ✅ 학습 시작
trainer.train()

# ✅ 평가 결과 출력
metrics = trainer.evaluate()
print(metrics)

# ------------------------------------------------------------
# ✅ 7) 테스트/추론 예시
# ------------------------------------------------------------
test_texts = [
    "정말 최고의 영화였다. 두 번 볼 거야!",
    "시간 아까움... 별로였다.",
]

# 테스트 문장 토크나이징
test_enc = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

# ---- 추론 ----
model.eval()
with torch.no_grad():
    # model.classifier[0].weight.device = 모델이 올라간 디바이스
    outputs = model(**{k: v.to(model.classifier[0].weight.device) for k,v in test_enc.items()})
    probs = torch.softmax(outputs["logits"], dim=-1).cpu().numpy()

# 결과 출력
for t, p in zip(test_texts, probs):
    print(f"{t} -> 긍정 확률={p[1]:.3f}, 부정 확률={p[0]:.3f}, 예측={np.argmax(p)}")


C:\Users\ekfla\AppData\Local\Temp\ipykernel_29588\1450293973.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.362800,0.356148,0.845000,0.845000
2,0.254800,0.374561,0.848125,0.848099
3,0.189800,0.414903,0.851875,0.851787


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.4149034023284912, 'eval_accuracy': 0.851875, 'eval_f1_macro': 0.8517869406002552, 'eval_runtime': 40.7068, 'eval_samples_per_second': 39.305, 'eval_steps_per_second': 0.614, 'epoch': 3.0}
정말 최고의 영화였다. 두 번 볼 거야! -> 긍정 확률=0.982, 부정 확률=0.018, 예측=1
시간 아까움... 별로였다. -> 긍정 확률=0.013, 부정 확률=0.987, 예측=0
